# 🛡️ VisionGuard Industrial — Dashboard de Control en Google Colab · v2.1

**Hito 2 | ResNet18 + GRU + NLP + supervisión humana**

Este cuaderno obtiene el historial de inspecciones desde Google Sheets (`Analisis_Integrado`) mediante tu propia sesión de Google. No requiere Streamlit ni archivos de claves JSON. **No publiques contraseñas ni claves privadas aquí.**

**Instrucciones:** ejecuta las celdas **1, 2 y 3** en ese orden. Google te pedirá autorización. El dashboard tiene filtros y botón para actualizar desde Sheets. En esta versión, los gráficos se muestran como imágenes estáticas para evitar que Colab corte la salida interactiva; los filtros sí son interactivos.

> **Alcance:** el historial corresponde al prototipo académico. Los pronósticos GRU son simulados, no datos reales de una planta. Las métricas del ResNet18 proceden de un conjunto de evaluación independiente de 715 imágenes.

In [ ]:
# 1. Instalar librerías en Colab (una vez por sesión)
!pip -q install gspread pandas matplotlib ipywidgets

In [ ]:
# 2. Conectar con el Google Sheets de VisionGuard usando TU sesión de Google
from google.colab import auth, output
from google.auth import default
import gspread
import pandas as pd

output.enable_custom_widget_manager()
auth.authenticate_user()
credentials, _ = default()
client = gspread.authorize(credentials)

SPREADSHEET_ID = 'TU_SPREADSHEET_ID'
WORKSHEET = 'Analisis_Integrado'

def cargar_inspecciones():
    ws = client.open_by_key(SPREADSHEET_ID).worksheet(WORKSHEET)
    rows = ws.get_all_records()
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    numeric_cols = ['confianza', 'pronostico_max_pct', 'casos_nlp']
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    df['fecha_registro'] = pd.to_datetime(df.get('fecha_registro'), errors='coerce')
    for c in ['clasificacion', 'prioridad', 'archivo', 'estado', 'linea_mayor_pronostico', 'diagnostico', 'recomendacion']:
        if c not in df.columns:
            df[c] = ''
        df[c] = df[c].fillna('').astype(str)
    if 'validacion_humana_realizada' not in df.columns:
        df['validacion_humana_realizada'] = False
    df['validacion_humana_realizada'] = df['validacion_humana_realizada'].astype(str).str.strip().str.lower().isin(['true','1','sí','si','yes','verdadero'])
    return df

inspecciones = cargar_inspecciones()
print(f'✅ Conectado a Google Sheets. Registros encontrados: {len(inspecciones)}')
print('🔒 Autenticación con Google Colab. No se necesitan claves JSON.')

In [ ]:
# 3. Dashboard Colab, versión robusta: gráficos estáticos + filtros interactivos
# Evita fig.show(renderer="colab") dentro de widgets.Output, que puede detener el renderizado.
import io, base64, html, traceback
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

AZUL = '#18395E'
CORAL = '#DD6660'
VERDE = '#179F8A'
NARANJA = '#E09A5E'
FONDO = '#F4F7FB'
LILA = '#7485CA'

plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':10, 'axes.titleweight':'bold'})

def esc(x):
    return html.escape(str(x) if x is not None else '')

def grafico_png(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=135, facecolor='white', bbox_inches='tight')
    plt.close(fig)
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii')

def tarjeta(titulo, valor, color=AZUL):
    return f'''<div style="flex:1 1 170px;background:white;border-radius:15px;border:1px solid #DBE4F0;padding:18px;box-shadow:0 4px 15px #172B4710">
      <div style="font:600 13px Arial;color:#62738A">{esc(titulo)}</div>
      <div style="font:800 29px Arial;color:{color};margin-top:8px">{esc(valor)}</div></div>'''

def titulo_seccion(num, nombre):
    return f'<h2 style="font:800 22px Arial;color:{AZUL};margin:27px 0 13px">{num} · {esc(nombre)}</h2>'

def imagen_tarjeta(uri, pie):
    return f'<div style="flex:1 1 360px;min-width:0;background:white;border:1px solid #DBE4F0;border-radius:14px;padding:12px"><div style="font:700 15px Arial;color:{AZUL};margin:7px 8px">{esc(pie)}</div><img alt="{esc(pie)}" src="{uri}" style="display:block;width:100%;max-height:340px;object-fit:contain"></div>'

def grafico_operaciones(d):
    fig,axs=plt.subplots(1,2,figsize=(12.5,3.65))
    clases=d['clasificacion'].value_counts()
    colores=[CORAL if str(z).strip().lower() == 'defectuosa' else VERDE for z in clases.index]
    axs[0].pie(clases.values,labels=clases.index,colors=colores,startangle=90,
               wedgeprops={'width':.36,'edgecolor':'white'},autopct='%1.0f%%',pctdistance=.83)
    axs[0].set_title('Clasificación de piezas',color=AZUL,pad=15)
    prioridades=d['prioridad'].value_counts()
    axs[1].bar(prioridades.index.astype(str), prioridades.values, color=NARANJA, width=.53)
    axs[1].set_title('Prioridad de inspección',color=AZUL,pad=15)
    axs[1].set_ylabel('Inspecciones')
    axs[1].set_ylim(0,max(1,int(prioridades.max())*1.25))
    axs[1].grid(axis='y', alpha=.15)
    axs[1].set_axisbelow(True)
    fig.tight_layout(w_pad=3)
    return grafico_png(fig)

def grafico_gru(d):
    disponibles=d.dropna(subset=['pronostico_max_pct'])
    if disponibles.empty:
        return None
    g=disponibles.groupby('linea_mayor_pronostico',dropna=False)['pronostico_max_pct'].mean().sort_values(ascending=False)
    fig,ax=plt.subplots(figsize=(12.5,3.1))
    barras=ax.bar(g.index.astype(str),g.values,color='#5575B9',width=.52)
    for b,value in zip(barras,g.values):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()+.2,f'{value:.2f}%',ha='center',va='bottom',fontweight='bold',color=AZUL)
    ax.set_title('Mayor pronóstico GRU por línea (promedio de inspecciones)',color=AZUL,pad=12)
    ax.set_xlabel('Línea destacada');ax.set_ylabel('Pronóstico simulado (%)')
    ax.set_ylim(0,max(10,float(g.max())*1.25));ax.grid(axis='y',alpha=.17);ax.set_axisbelow(True)
    fig.tight_layout()
    return grafico_png(fig)

def grafico_estados(d):
    g=d['estado'].value_counts()
    fig,ax=plt.subplots(figsize=(12.5,max(2.4,0.5*len(g)+1.6)))
    ax.barh(range(len(g)),g.values,color=VERDE,height=.54)
    ax.set_yticks(range(len(g)), [str(s)[:50] for s in g.index])
    ax.set_xlim(0,max(g.values)*1.3)
    ax.invert_yaxis();ax.grid(axis='x',alpha=.17);ax.set_axisbelow(True)
    ax.set_xlabel('Número de inspecciones')
    ax.set_title('Estado y trazabilidad de decisiones',pad=12,color=AZUL)
    fig.tight_layout()
    return grafico_png(fig)

def matriz_resnet():
    fig,ax=plt.subplots(figsize=(6.5,3.8))
    cm=np.array([[259,3],[13,440]])
    ax.imshow(cm,cmap='Blues',vmin=0,vmax=440)
    for (i,j),v in np.ndenumerate(cm):
        ax.text(j,i,str(v),ha='center',va='center',fontweight='bold',fontsize=15,color='white' if v>210 else AZUL)
    ax.set_xticks([0,1],['Pred. correcta','Pred. defectuosa'])
    ax.set_yticks([0,1],['Real correcta','Real defectuosa'])
    ax.set_title('Matriz de confusión · ResNet18',pad=12,color=AZUL)
    fig.tight_layout()
    return grafico_png(fig)

filtro_clase=widgets.Dropdown(description='Clasificación:',style={'description_width':'initial'},layout=widgets.Layout(width='285px'))
filtro_prioridad=widgets.Dropdown(description='Prioridad:',style={'description_width':'initial'},layout=widgets.Layout(width='240px'))
boton_actualizar=widgets.Button(description='Actualizar datos',button_style='info',icon='refresh',layout=widgets.Layout(width='185px'))
salida=widgets.Output(layout=widgets.Layout(width='100%'))
_en_cambio = False

def sincronizar_filtros():
    global _en_cambio
    _en_cambio=True
    try:
        clases=sorted(inspecciones['clasificacion'].dropna().astype(str).unique().tolist()) if not inspecciones.empty else []
        prioridades=sorted(inspecciones['prioridad'].dropna().astype(str).unique().tolist()) if not inspecciones.empty else []
        clase_ant,prioridad_ant=filtro_clase.value,filtro_prioridad.value
        filtro_clase.options=['Todas']+clases
        filtro_prioridad.options=['Todas']+prioridades
        filtro_clase.value=clase_ant if clase_ant in filtro_clase.options else 'Todas'
        filtro_prioridad.value=prioridad_ant if prioridad_ant in filtro_prioridad.options else 'Todas'
    finally:
        _en_cambio=False

def dibujar(_=None):
    if _en_cambio:
        return
    with salida:
        clear_output(wait=True)
        try:
            if inspecciones.empty:
                display(HTML('<div style="padding:20px;background:#FFF4E7">La hoja no contiene inspecciones todavía.</div>'))
                return
            d=inspecciones.copy()
            if filtro_clase.value!='Todas':
                d=d[d['clasificacion']==filtro_clase.value]
            if filtro_prioridad.value!='Todas':
                d=d[d['prioridad']==filtro_prioridad.value]
            if d.empty:
                display(HTML('<div style="padding:20px;background:#FFF4E7">Ninguna inspección coincide con los filtros.</div>'))
                return
            total=len(d)
            defectos=int(d['clasificacion'].astype(str).str.strip().str.casefold().eq('defectuosa').sum())
            revisadas=int(d['validacion_humana_realizada'].sum())
            confianza=pd.to_numeric(d['confianza'],errors='coerce').mean()
            if pd.notna(confianza) and confianza>1: confianza/=100
            texto_conf=f'{confianza*100:.2f}%' if pd.notna(confianza) else '—'
            cabecera=f'''<div style="background:linear-gradient(110deg,#173657,#275783);border-radius:17px;padding:28px;margin:6px 0 15px">
              <div style="font:700 11px Arial;color:#FFC79A;letter-spacing:2px">VISIONGUARD INDUSTRIAL · HITO 2 · COLAB v2.1</div>
              <div style="font:800 29px Arial;color:white;margin:9px 0">Centro de Control Inteligente</div>
              <div style="font:14px Arial;color:#E1EAF4">Visión artificial · ResNet18 · GRU · NLP · Supervisión humana</div></div>'''
            cartas=''.join([tarjeta('Inspecciones',total),tarjeta('Piezas defectuosas',defectos,CORAL),
                     tarjeta('Tasa de defectos',f'{defectos/total*100:.1f}%',CORAL),
                     tarjeta('Revisiones humanas',revisadas,VERDE),tarjeta('Confianza promedio',texto_conf)])
            bloques=[cabecera, f'<div style="display:flex;flex-wrap:wrap;gap:12px;background:{FONDO};border-radius:16px;padding:12px">{cartas}</div>']
            bloques.append(titulo_seccion('01','Operaciones e inspección'))
            bloques.append(imagen_tarjeta(grafico_operaciones(d),'Distribución de clasificaciones y prioridades'))
            bloques.append(titulo_seccion('02','Predicción GRU y trazabilidad'))
            pron=grafico_gru(d)
            bloques.append(imagen_tarjeta(pron,'Pronósticos por línea') if pron else '<p>No hay valores de pronóstico GRU en las inspecciones filtradas.</p>')
            bloques.append('<p style="color:#83551F;background:#FFF3DD;border-radius:9px;padding:12px;font:13px Arial">⚠️ Pronósticos GRU simulados: no son mediciones de calidad ni tasas reales de defectos.</p>')
            bloques.append(titulo_seccion('03','Decisiones humanas'))
            bloques.append(imagen_tarjeta(grafico_estados(d),'Distribución del estado de las inspecciones'))
            autorizadas=int(d['estado'].str.contains('AUTORIZADA_POR_HUMANO',case=False,na=False).sum())
            rechazadas=int(d['estado'].str.contains('RECHAZADA_POR_HUMANO',case=False,na=False).sum())
            pendientes=int((~d['validacion_humana_realizada']).sum())
            bloques.append('<div style="display:flex;gap:12px;flex-wrap:wrap">'+tarjeta('Acciones autorizadas',autorizadas,VERDE)+tarjeta('Acciones rechazadas',rechazadas,CORAL)+tarjeta('Pendientes',pendientes)+'</div>')
            bloques.append(titulo_seccion('04','Historial y diagnóstico NLP'))
            cols=[c for c in ['fecha_registro','archivo','clasificacion','confianza','prioridad','linea_mayor_pronostico','pronostico_max_pct','estado','validacion_humana_realizada'] if c in d.columns]
            tabla=d[cols].sort_values('fecha_registro',ascending=False).copy()
            # Evitar mostrar HTML sin escapar de la hoja, y evitar tablas desbordadas.
            bloques.append('<div style="overflow-x:auto;background:white;padding:10px;border-radius:12px">'+tabla.to_html(index=False,escape=True,border=0,classes='vgtable')+'</div>')
            ultimo=d.sort_values('fecha_registro',ascending=False).iloc[0]
            diag=esc(ultimo.get('diagnostico','') or 'Sin diagnóstico registrado')
            reco=esc(ultimo.get('recomendacion','') or 'Sin recomendación registrada')
            bloques.append(f'<div style="background:#EFF5FB;border-radius:11px;padding:17px;color:{AZUL};font:14px/1.7 Arial"><b>Última inspección: {esc(ultimo.get("archivo",""))}</b><p><b>Diagnóstico IA:</b> {diag}</p><p><b>Recomendación:</b> {reco}</p></div>')
            bloques.append(titulo_seccion('05','Evidencia experimental ResNet18'))
            bloques.append('<p style="color:#42576E;font:14px Arial">Evaluación independiente con 715 imágenes (no corresponde a estas inspecciones).</p>')
            bloques.append('<div style="display:flex;gap:12px;flex-wrap:wrap">'+''.join([
                tarjeta('Accuracy','97,76 %'),tarjeta('Precision · defecto','99,32 %'),
                tarjeta('Recall · defecto','97,12 %'),tarjeta('F1 · defecto','98,21 %')])+'</div>')
            bloques.append(imagen_tarjeta(matriz_resnet(),'Matriz de confusión del modelo ResNet18'))
            bloques.append('<p style="font:12px Arial;color:#68809A;margin:24px 0">Prototipo académico VisionGuard Industrial · Las decisiones críticas requieren inspección humana.</p>')
            estilo_html='''<style>.vgtable{width:100%;border-collapse:collapse;color:#173458;font:12px Arial}.vgtable th{background:#EDF2F9;padding:10px;text-align:left;white-space:nowrap}.vgtable td{border-bottom:1px solid #DDE5EF;padding:8px;white-space:nowrap}</style>'''
            display(HTML(estilo_html+'<div style="background:#F4F7FB;padding:12px 16px;border-radius:14px">'+''.join(bloques)+'</div>'))
        except Exception as exc:
            # Mensajes breves y sin revelar contenido secreto de credenciales.
            display(HTML(f'<div style="background:#FFF2F1;color:#9A2F27;padding:14px;border-radius:10px">Error al mostrar el dashboard: {esc(type(exc).__name__)}</div>'))
            raise

def actualizar(_):
    global inspecciones
    try:
        inspecciones=cargar_inspecciones()
        sincronizar_filtros()
        dibujar()
    except Exception as exc:
        with salida:
            clear_output(wait=True)
            display(HTML(f'<p style="color:#AE3737">No fue posible actualizar la hoja: {esc(type(exc).__name__)}</p>'))

sincronizar_filtros()
filtro_clase.observe(dibujar,names='value')
filtro_prioridad.observe(dibujar,names='value')
boton_actualizar.on_click(actualizar)
controles=widgets.HBox([filtro_clase,filtro_prioridad,boton_actualizar],layout=widgets.Layout(flex_flow='row wrap',gap='10px'))
display(controles,salida)
dibujar()
